[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
import torch

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    logits /= temperature
    if top_k > 0:
        vals,idx = logits.topk(top_k,dim=-1)
        scores = torch.full_like(logits, float("-inf"))
        scores.scatter_(dim=-1,index=idx,src=vals)
        probs = torch.softmax(scores,dim=-1)
    if top_p < 1.0:
        probs = torch.softmax(logits,dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True,dim=-1)
        cum_probs=torch.cumsum(sorted_probs,dim=-1)
        mask = (cum_probs < top_p).float()
        scores = torch.full_like(logits, float("-inf"))
        scores.scatter_(dim=-1,index=sorted_indices,src=sorted_probs*mask)
        probs = torch.softmax(scores,dim=-1)
        
    return torch.multinomial(probs,num_samples=1).item()

In [11]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 3
top_p=0.5: 0
temp=0.01: 3


In [12]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ❌ [1/4] top_k=1 always returns argmax
     top_k=1 should return argmax
  ❌ [2/4] Low temperature concentrates
     Low temp should pick argmax, got [33, 35, 32]
  ✅ [3/4] All tokens reachable (no filtering) (66.0ms)
  ✅ [4/4] Returns valid index (4.4ms)
──────────────────────────────────────────────────
  📊 2/4 tests passed.
  Keep going! Use hint("topk_sampling") if you're stuck.

